In [ ]:
raw_path = (
    "abfss://raw@stcarrentaldev.dfs.core.windows.net/"
    "cars/CARS.parquet"
)

cars = spark.read.parquet(raw_path)

display(cars)
print(f"Rows: {cars.count()}")
cars.printSchema()

In [ ]:
from pyspark.sql import functions as F

missing_plates = cars.filter(
    F.col("plate").isNull() | (F.trim(F.col("plate")) == "")
).count()

duplicate_plates = (
    cars.groupBy("plate")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Missing plate: {missing_plates}")
print(f"Duplicate plate: {duplicate_plates}")

In [ ]:
cars_clean = (
    cars
    .withColumnRenamed("registrationdate" , "registration_date")
    .withColumn("processed_at", F.current_timestamp()) 
)
display(cars_clean) 

In [ ]:
curated_path = ( 
    "abfss://curated@stcarrentaldev.dfs.core.windows.net/cars"
)

(
    cars_clean.write
    .format("delta")
    .mode("overwrite")
    .save(curated_path)
)

In [ ]:
saved_cars= spark.read.format("delta").load(curated_path)

raw_count = cars.count()
saved_count = saved_cars.count()

print(f"Raw rows: {raw_count}")
print(f"Curated rows: {saved_count}")

assert raw_count == saved_count, "Row count do not  match" 
display(saved_cars)

In [ ]:
%sql 
SELECT current_catalog(), current_schema();

In [ ]:
%sql 
USE CATALOG dbw_carrental_dev_7405614156846449;

CREATE SCHEMA IF NOT EXISTS curated;

CREATE TABLE IF NOT EXISTS curated.cars

USING DELTA 

LOCATION 'abfss://curated@stcarrentaldev.dfs.core.windows.net/cars';
SELECT * FROM curated.cars LIMIT 10;


In [ ]:
from pyspark.sql import functions as F 

from pyspark.sql import functions as F

table_keys = {
    "rental_offices": ["officename"],
    "cars": ["plate"],
    "have_optional": ["rentalid"],
    "drivers": ["licensenumber"],
    "rentals": ["plate", "pickupdate"],
    "drive": ["licensenumber", "plate", "pickupdate"],
    "insurances": ["risk", "plate", "pickupdate"],
    "payments": ["plate", "pickupdate"],
}

print(f"Tables configured: {len(table_keys)}")